# Differentiable Simulation for Parameter Estimation

In this tutorial, we learn how to estimate initial conditions of a 1D wave equation by combining a finite-difference time-domain (FDTD) solver with automatic differentiation.

## Environment Setup

First, install the python dependencies and initialize the environment. You can use any package you like for auto-diff (e.g., PyTorch, JAX, TensorFlow, etc.), but for this demo we will use [Taichi](https://www.taichi-lang.org/).

In [ ]:
# Install and import dependencies
!pip install taichi -q
import taichi as ti
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Initialize
np.random.seed(12345)
try:
    ti.init(arch=ti.gpu)
except:
    ti.init(arch=ti.cpu)

## The 1D Wave Equation and FDTD Discretization

Consider the continuous wave equation:
$$\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}$$

over time $t\in T$ and space $x\in X$.

Discretized via central differences gives:

$$u_\texttt{x}^{\texttt{t}+1} = 2u_\texttt{x}^\texttt{t} - u_\texttt{x}^{\texttt{t}-1} + \left(c\frac{\Delta t}{\Delta x}\right)^2 (u_{\texttt{x}+1}^\texttt{t} - 2u_\texttt{x}^\texttt{t} + u_{\texttt{x}-1}^\texttt{t}).$$


In this tutorial we will try to solve the following problem:
- Given *partial* observations on $u_{\texttt{x}}^{\texttt{t}}$ for $\texttt{x}\in \texttt{x}_{\texttt{obs}}\subseteq X$ and $\texttt{t}\in \texttt{t}_{\texttt{obs}}\subseteq T$
- Find the initial conditions (IC) $u_\texttt{x}^0$ and $u_\texttt{x}^1$
- To recover the partial observation at $\texttt{x}\in \texttt{x}_{\texttt{obs}}$ and $\texttt{t}\in \texttt{t}_{\texttt{obs}}$.

So, this problem is an inverse problem: finding initial conditions from distorted observations. To solve this problem, we will apply gradient backpropagation through differentiable FDTD simulations.

First, define the physical quantities as variables.

In [ ]:
# Simulation parameters
nx = 200    # Number of spatial grid points
nt = 500    # Number of time steps
dx = 0.01   # Spatial step size
dt = 0.005  # Time step size
c  = 1.0    # Wave speed

# Time stepping using FDTD
r_squared = (c * dt / dx) ** 2

These are all the variables needed to perform FDTD on the above wave equation! Now, all we need to do is write the FDTD recursion function to run the simulation. However, since we'll be performing gradient backpropagation in addition to the simulation, we'll add a few more definitions for this optimization.

In [ ]:
# Observation parameters
n_obs = 3             # Number of observation points
obs_start_time = 100  # Start observing after some initial time
obs_end_time = 300    # End observation time

assert obs_end_time <= nt, "Observation time must be less than total time"
assert obs_start_time < obs_end_time, "Observation start time must be before observation end time"
assert obs_start_time > 0, "Observation start time must be greater than 0"

# Optimization parameters
iterations = 300
learning_rate_u = 0.01
learning_rate_v = 0.50 # set lr for v higher than u, as the velocity will be scaled by dt
grad_clip = 1.0

In [ ]:
# Fields with gradients enabled for optimization variables
u = ti.field(ti.f32, shape=(nx, nt), needs_grad=True)    # Displacement field u(x,t)
initial_u = ti.field(ti.f32, shape=nx, needs_grad=True)  # Initial displacement
initial_v = ti.field(ti.f32, shape=nx, needs_grad=True)  # Initial velocity

# Target trajectory at observation point
target_trajectory = ti.field(ti.f32, shape=(n_obs, nt))
obs_index = ti.field(ti.i32, shape=n_obs)  # Indices of observation points

# Loss
loss = ti.field(ti.f32, shape=(), needs_grad=True)

# To be filled with random observation points
x_obs = [0] * n_obs

# Target config (Gaussian-type IC)
target_center = None # Target IC's peak center; Set to `None` to randomize
target_width  = None # Target IC's peak width;  Set to `None` to randomize

As we recall, the FDTD recursion relation is:
$$u_\texttt{x}^{\texttt{t}+1} = 2u_\texttt{x}^\texttt{t} - u_\texttt{x}^{\texttt{t}-1} + \left(c\frac{\Delta t}{\Delta x}\right)^2 (u_{\texttt{x}+1}^\texttt{t} - 2u_\texttt{x}^\texttt{t} + u_{\texttt{x}-1}^\texttt{t}).$$

Implementing this in a differentiable function is very straightforward. Simply write the update expression above inside the for loop. The decorator `@ti.kernel` put in front of a function, will indicate Taichi to automatically parallelize this sequential operation inside the function.


In [ ]:
@ti.kernel
def fdtd_step(t: ti.i32):
    """Perform a single FDTD step"""
    for i in range(1, nx - 1):
        u[i, t] = 2.0 * u[i, t - 1] - u[i, t - 2] \
            + r_squared * (u[i + 1, t - 1] - 2.0 * u[i, t - 1] + u[i - 1, t - 1])

Additionally, we need to specify conditions at the boundary (the edge of the string). Since a typical string is always fixed at both ends, we'll simply assign a displacement of 0 as the Dirichlet boundary condition.

$$ u_0^\texttt{t}=0 \qquad \text{and} \qquad u_{n_x-1}^\texttt{t}=0 $$

for $\texttt{t}=0,1,...,n_t$. Also, we will write a function that copies the (to-be-optimized) initial conditions on $u$.

In [ ]:
@ti.kernel
def assign_ic():
    # Initialize first two time steps
    for i in range(nx):
        u[i, 0] = initial_u[i]
        if i == 0 or i == nx - 1:
            u[i, 1] = 0.0  # Boundary conditions
        else:
            # Second time step using initial velocity
            u[i, 1] = initial_u[i] + dt * initial_v[i]

@ti.kernel
def apply_bc(t: ti.i32):
    # Apply boundary conditions (fixed ends)
    u[0, t] = 0.0
    u[nx - 1, t] = 0.0

def fdtd_simulation():
    """Perform the complete FDTD simulation"""
    assign_ic()
    for t in range(1, nt - 1):
        fdtd_step(t + 1)
        apply_bc(t + 1)

Now if we have a pair of initial conditions, we can see the FDTD results.

In [ ]:
def initialize_guess(center=None, width=None):
    """Initialize the initial condition guess"""
    # Start with a simple Gaussian pulse positioned to potentially reach the observation point
    center_pos = 2 * np.random.rand() if center is None else center
    width = 0.1 * np.random.rand() if width is None else width

    initial_u_data = np.zeros(nx, np.float32)
    initial_v_data = np.zeros(nx, np.float32)

    for i in range(nx):
        x = i * dx
        # Gaussian pulse
        initial_u_data[i] = 0.2 * np.exp(-((x - center_pos) / width) ** 2)
        # Small initial velocity toward the observation point
        initial_v_data[i] = 0.5 * np.exp(-((x - center_pos) / width) ** 2)# * np.sign(x_obs - x)

    initial_u.from_numpy(initial_u_data)
    initial_v.from_numpy(initial_v_data)

In [ ]:
initialize_guess()
fdtd_simulation()

u_data = u.to_numpy()
x_coords = np.linspace(0, (nx-1) * dx, nx)
t_coords = np.linspace(0, (nt-1) * dt, nt)
T, X = np.meshgrid(t_coords, x_coords)

levels = np.linspace(u_data.min(), u_data.max(), 50)
plt.figure(figsize=(10, 3))
contour = plt.contourf(T, X, u_data, levels=levels, cmap='RdBu_r')
plt.xlabel('Time t')
plt.ylabel('Position x')
plt.title('Wave Propagation')
plt.colorbar(contour)

Below is a function that simulates a ground truth by
1. Defining the GT initial condition
2. Computing the FDTD simulation
3. Saving the result as `target_trajectory` array.

In [ ]:
def create_target_trajectory(
        simulated_args : dict = {
            'pulse_center': 0.5,
            'pulse_width': 0.05,
        },
        observation_preset : list = None,
    ):
    """Create a target trajectory for the observation point"""
    target_data = np.zeros((n_obs, nt), dtype=np.float32)
    obs_index_data = np.zeros(n_obs, dtype=np.int32)

    initialize_guess(center=simulated_args['pulse_center'], width=simulated_args['pulse_width'])
    fdtd_simulation()

    if observation_preset is not None:
        assert len(observation_preset) == n_obs, observation_preset
    for n in range(n_obs):
        random_obs = 0.6 * np.random.rand() + 0.2 if observation_preset is None else observation_preset[n]
        obs_index_data[n] = int(random_obs * nx)
        x_obs[n] = obs_index_data[n] * dx
        for t in range(nt):
            target_data[n,t] = u[obs_index_data[n], t]

    obs_index.from_numpy(obs_index_data)
    target_trajectory.from_numpy(target_data)

Now, that's it! We implemented the FDTD simulation using a differentiable package (Taichi in this case), and gradient backpropagation will be computed automatically in the backend.

## Helper Functions

For plotting and saving results.

In [ ]:
import os
import glob
import imageio
from taichi.tools.video import VideoManager
from IPython.display import clear_output

def visualize_results(is_final=False, global_minmax=None):
    """Visualize the optimization results"""
    # Run final simulation to get results
    assign_ic()
    fdtd_simulation()
    compute_loss()

    clear_output(wait=True)
    nrows = n_obs + 1
    fig, ax = plt.subplots(nrows=nrows, ncols=2, figsize=(17, 3 * nrows))

    # Plot 1: Optimized initial conditions
    x_coords = np.linspace(0, (nx-1) * dx, nx)
    initial_u_data = initial_u.to_numpy()
    initial_v_data = initial_v.to_numpy()

    ax[0,0].plot(x_coords, initial_u_data, 'b-', linewidth=2, label='Initial displacement u(x,0)')
    ax00_twin = ax[0,0].twinx()
    ax00_twin.plot(x_coords, initial_v_data, 'r--', linewidth=2, label='Initial velocity ∂u/∂t(x,0)')
    for n in range(n_obs):
        ax[0,0].axvline(x=x_obs[n], color='yellow', linestyle='--', linewidth=2, label=f'Observation point')
    ax[0,0].set_xlabel('Position x')
    ax[0,0].set_ylabel('Displacement', color='b')
    ax00_twin.set_ylabel('Velocity', color='r')
    ax[0,0].set_title('Optimized Initial Conditions')
    ax[0,0].grid(True)
    ax[0,0].legend(loc='upper left')
    ax00_twin.legend(loc='upper right')

    # Plot 2: Wave evolution (space-time contour)
    u_data = u.to_numpy()
    t_coords = np.linspace(0, (nt-1) * dt, nt)
    T, X = np.meshgrid(t_coords, x_coords)
    if global_minmax is not None:
        levels = np.linspace(global_minmax[0], global_minmax[1], 50)
    else:
        levels = np.linspace(u_data.min(), u_data.max(), 50)
    contour = ax[0,1].contourf(T, X, u_data, levels=levels, cmap='RdBu_r')
    for n in range(n_obs):
        ax[0,1].axhline(y=x_obs[n], color='yellow', linestyle='--', linewidth=2, label=f'Observation point (x={x_obs[n]:.2f})')
    ax[0,1].axvline(x=obs_start_time * dt, color='lime', linestyle=':', alpha=0.8, label='Obs window start')
    ax[0,1].axvline(x=obs_end_time * dt, color='lime', linestyle=':', alpha=0.8, label='Obs window end')
    ax[0,1].set_xlabel('Time t')
    ax[0,1].set_ylabel('Position x')
    ax[0,1].set_title('Wave Propagation')
    ax[0,1].legend(loc='lower right')
    plt.colorbar(contour, ax=ax[0,1], label='Displacement u(x,t)')

    # Plot 3: Trajectory comparison at observation point
    obs_index_data = obs_index.to_numpy()
    for n in range(n_obs):
        simulated_trajectory = u_data[obs_index_data[n], :]
        target_traj = target_trajectory.to_numpy()[n, :]

        ax[n+1,0].plot(t_coords, target_traj, 'r-', linewidth=3, label='Target trajectory', alpha=0.8)
        ax[n+1,0].plot(t_coords, simulated_trajectory, 'b--', linewidth=2, label='Simulated trajectory')

        # Highlight observation window
        obs_window_t = t_coords[obs_start_time:obs_end_time]
        obs_window_target = target_traj[obs_start_time:obs_end_time]
        obs_window_sim = simulated_trajectory[obs_start_time:obs_end_time]

        ax[n+1,0].fill_between(obs_window_t, obs_window_target, obs_window_sim,
                         alpha=0.3, color='gray', label='Error region (obs window)')

        ax[n+1,0].set_xlabel('Time t')
        ax[n+1,0].set_ylabel('Displacement')
        ax[n+1,0].set_title(f'Trajectory Matching at x = {x_obs[n]:.2f}')
        ax[n+1,0].grid(True, alpha=0.3)
        ax[n+1,0].legend()

        # Plot 4: Error analysis
        error = simulated_trajectory - target_traj
        rmse_full = np.sqrt(np.mean(error**2))
        rmse_obs = np.sqrt(np.mean(error[obs_start_time:obs_end_time]**2))

        ax[n+1,1].plot(t_coords, error, 'g-', linewidth=2, label='Error (sim - target)')
        ax[n+1,1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax[n+1,1].axvspan(obs_start_time * dt, obs_end_time * dt, alpha=0.2, color='yellow',
                   label='Observation window')
        ax[n+1,1].set_xlabel('Time t')
        ax[n+1,1].set_ylabel('Error')
        ax[n+1,1].set_title(f'Error Analysis\nFull RMSE: {rmse_full:.4f}, Obs RMSE: {rmse_obs:.4f}')
        ax[n+1,1].grid(True, alpha=0.3)
        ax[n+1,1].legend()

    plt.tight_layout()
    plt.show()

    if is_final:
        # Print summary
        print(f"\n=== Optimization Summary ===")
        print(f"Final loss: {loss[None]:.6f}")
        print(f"RMSE over full time: {rmse_full:.6f}")
        print(f"RMSE over observation window: {rmse_obs:.6f}")
        print(f"Target trajectory energy: {np.sum(target_traj**2):.4f}")
        print(f"Initial condition energy: {np.sum(initial_u_data**2):.4f}")



## Parameter Estimation with Gradient Backprop through Differentiable Simulation

To estimate the parameters (the IC herein) using gradient backpropagation, we first need to define the loss which we take the gradient of. Since we assume that we only have our partial observation, we define our loss as the standard MSE loss over $\texttt{t}\in\texttt{t}_\texttt{obs}$ and $\texttt{x}\in\texttt{x}_\texttt{obs}$.

To update the IC through the gradient descent, it's enough to simply subtract the ICs' gradients. Optionally, we can take the gradient clipping but it's not that critical here.

In [ ]:
@ti.kernel
def compute_loss():
    """Compute loss based on trajectory matching at observation point"""
    for t in range(obs_start_time, obs_end_time):
        for n in range(n_obs):
            diff = u[obs_index[n], t] - target_trajectory[n, t]
            loss[None] += diff * diff

@ti.kernel
def apply_grad():
    # gradient descent
    for i in initial_u.grad:
        initial_u[i] -= ti.max(ti.min(learning_rate_u * initial_u.grad[i], grad_clip), -grad_clip)
        initial_v[i] -= ti.max(ti.min(learning_rate_v * initial_v.grad[i], grad_clip), -grad_clip)

We mark the initial displacement and velocity as differentiable fields. Using Taichi's `ti.ad.Tape`, we record all operations in the forward FDTD pass and compute gradients of a loss function (mismatch at observation points) with respect to the initial conditions.

In [ ]:
def optimize_initial_condition():
    """Optimize the initial condition using gradient descent"""

    print("Starting optimization...")

    for iter in range(iterations):
        # Reset loss
        loss[None] = 0.0

        # Forward simulation and loss computation
        with ti.ad.Tape(loss):
            fdtd_simulation()
            compute_loss()

        if iter % 20 == 0:
            visualize_results()

        apply_grad()

Now we're ready to optimize the IC! Running the `optimize_initial_condition()` will update values of the taichi variables defined in [here](https://colab.research.google.com/drive/17252KH_y_ypOXwXLEYGmDZQk5LFTMT2B#scrollTo=Ie0Q2Oy5cnns&line=5&uniqifier=1) using the gradient descent. You can run the optimization right away, but let's just see how our target and the initial guess looks like.

In [ ]:
print("=== 1D Wave Equation Inverse Design ===")
print(f"Grid points: {nx}, Time steps: {nt}")
print(f"dx = {dx}, dt = {dt}, c = {c}")
print(f"CFL number: {c * dt / dx:.3f}")

# Create target trajectory
create_target_trajectory(
    simulated_args = {
        'pulse_center': target_center,
        'pulse_width': target_width,
    },
    observation_preset = np.linspace(0.1, 0.7, n_obs) if n_obs > 1 else [0.5],
)
# Save GT trajectory (just for visualization)
gt_data = u.to_numpy()

# Initialize initial condition guess
initialize_guess()

print(f"Observation point: x = {x_obs}")
print(f"Observation time window: t = {obs_start_time * dt:.3f} to {obs_end_time * dt:.3f}")

# Run initial simulation to see the baseline
assign_ic()
fdtd_simulation()
compute_loss()
print(f"Initial loss: {loss[None]:.6f}")

# Plot the initial guess along with the ground truth data
u_data = u.to_numpy()
x_coords = np.linspace(0, (nx-1) * dx, nx)
t_coords = np.linspace(0, (nt-1) * dt, nt)
T, X = np.meshgrid(t_coords, x_coords)

global_min = min(u_data.min(), gt_data.min())
global_max = max(u_data.max(), gt_data.max())
levels = np.linspace(global_min, global_max, 50)

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(17, 2))
contour1 = ax[0].contourf(T, X,  u_data, levels=levels, cmap='RdBu_r')
contour2 = ax[1].contourf(T, X, gt_data, levels=levels, cmap='RdBu_r')
for i in range(2):
  ax[i].set_xlabel('Time t')
  ax[i].set_ylabel('Position x')
ax[0].set_title('$u$ with Initial Guess')
ax[1].set_title('$u$ with GT Initial Condition')
plt.colorbar(contour1)
plt.colorbar(contour2)

In [ ]:
optimize_initial_condition()       # Optimize the initial condition
visualize_results(is_final=True, global_minmax=(global_min, global_max))   # Visualize results

## Running your own simulation

You can run your own simulations with different choice of values for the variables defined [here](https://colab.research.google.com/drive/17252KH_y_ypOXwXLEYGmDZQk5LFTMT2B#scrollTo=B-_rePPbcUzv&line=7&uniqifier=1). Here are a list of variables what you might want to change:

- Observation parameters
  - `n_obs`
    - Try increasing/decreasing the number of observations.
    - More observations can result in better convergence to some extent.
  - `obs_start_time` / `obs_end_time`
    - Try adjusting the range of the observation time window.
    - Longer observation window can narrow down the solution closer to the GT.

- Optimization parameters
  - `iterations`
    - Try increasing the number of iterations.
    - It is usually advised to set 1000+ number of steps.
